# PG Safety and Price Intelligence for Bengaluru
### End-to-End Data Science & Machine Learning Workflow

---
### Abstract
This project provides a comprehensive data-driven decision-support workflow for individuals evaluating Paying Guest (PG) accommodations in Bengaluru. It systematically analyzes listing attributes such as locality, rent, deposit, room sharing type, and amenities, estimates fair market rent using machine learning (Linear Regression baseline vs. Random Forest Regressor), surfaces transparent, explainable warning indicators for listings requiring manual verification, and delivers a multi-criteria recommendation engine tailored to user preferences and budget.

> **Important Note & Ethical Disclaimer:** Warning indicators are prompts for independent verification—not certified proof of fraud or scams. Any numerical outputs derived from the demonstration dataset serve illustrative and analytical purposes.

In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src directory to path
sys.path.append(os.path.abspath('../src'))
from data_cleaner import clean_dataset, clean_currency, normalize_locality
from feature_pipeline import create_feature_preprocessor, prepare_feature_dataframe, ALL_MODEL_FEATURES
from model_trainer import train_and_evaluate, predict_rent_estimate
from warning_engine import evaluate_listing_safety
from recommender import PGRecommender

sns.set_theme(style='whitegrid', font_scale=1.05)
print('Environment initialized successfully.')

## 1. Data Understanding & Ingestion
We load the demonstration dataset containing 400 listing records across major Bengaluru tech corridors and residential hubs.

In [2]:
raw_data_path = '../data/pg_listings.csv'
raw_df = pd.read_csv(raw_data_path)
print(f'Raw Dataset Shape: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns')

# Check missing value profile
missing = raw_df.isnull().sum()
print('\nMissing values detected:')
print(missing[missing > 0])
raw_df.head(3)

## 2. Data Cleaning & Normalization
Real-world rental data frequently suffers from inconsistent currency formatting (e.g., `₹13,300`, `46,500 INR`), case discrepancies in localities (e.g., `WHITEFIELD` vs `Whitefield`), missing amenities, and unstandardized deposit values.

Our cleaning pipeline handles:
1. **Currency Parsing**: Stripping currency symbols, commas, and converting to numeric float.
2. **Locality Standardization**: Normalizing Bengaluru localities to canonical title case.
3. **Missing Value Imputation**: Median deposit by locality/room-type, mode for categorical amenities.
4. **Multi-Hot Vectorization**: Decomposing semicolon-separated amenities into binary flags.

In [3]:
cleaned_df, summary = clean_dataset(raw_df)
print('Cleaning Summary:')
for k, v in summary.items():
    if isinstance(v, float):
        print(f' - {k}: {v:,.2f}')
    else:
        print(f' - {k}: {v}')

print(f'\nCleaned DataFrame Shape: {cleaned_df.shape}')

## 3. Exploratory Data Analysis & Visualizations
We examine how rent varies by locality and room sharing configuration, inspect deposit-to-rent ratios, and analyze amenity coverage.

In [4]:
# 1. Rent Distribution by Locality
plt.figure(figsize=(10, 6))
loc_order = cleaned_df.groupby('locality')['monthly_rent_inr'].median().sort_values().index
sns.boxplot(data=cleaned_df, y='locality', x='monthly_rent_inr', order=loc_order, palette='mako')
plt.title('Monthly Rent Distribution Across Bengaluru Localities (INR)', fontweight='bold')
plt.xlabel('Monthly Rent (INR)')
plt.show()

# 2. Deposit-to-Rent Ratio
plt.figure(figsize=(8, 4.5))
sns.histplot(cleaned_df['deposit_to_rent_ratio'], kde=True, bins=25, color='#0284c7')
plt.axvline(4.5, color='#ef4444', linestyle='--', label='High Deposit Alert Threshold (4.5x)')
plt.title('Deposit-to-Rent Ratio Distribution', fontweight='bold')
plt.xlabel('Deposit / Rent Ratio')
plt.legend()
plt.show()

## 4. Machine Learning Modeling: Rent Prediction
We formulate monthly rent estimation as a supervised regression task. As required by the project specifications, we avoid target leakage by splitting the training and test sets (80% train, 20% test) before model training.

- **Candidate Features**: `locality`, `room_type`, `food_option`, `wifi`, `ac`, `bathroom_type`, and individual amenity binary flags.
- **Baseline Model**: Linear Regression (interpretable linear benchmark).
- **Non-linear Comparison**: Random Forest Regressor (captures non-linear interactions between locality, room type, and facilities).
- **Evaluation Metrics**: MAE (rupees error), RMSE (penalizes large outliers), and $R^2$ (variance explained).

In [5]:
rf_model, lr_model, eval_metrics, test_eval_df = train_and_evaluate(cleaned_df)

eval_table = pd.DataFrame([
    {
        'Model': 'Linear Regression (Baseline)',
        'Test MAE (INR)': eval_metrics['linear_regression']['test']['mae'],
        'Test RMSE (INR)': eval_metrics['linear_regression']['test']['rmse'],
        'Test R2 Score': eval_metrics['linear_regression']['test']['r2']
    },
    {
        'Model': 'Random Forest Regressor',
        'Test MAE (INR)': eval_metrics['random_forest']['test']['mae'],
        'Test RMSE (INR)': eval_metrics['random_forest']['test']['rmse'],
        'Test R2 Score': eval_metrics['random_forest']['test']['r2']
    }
])

print('\nHeld-out Test Set Model Comparison:')
eval_table

## 5. Residual Analysis & Segment Error Inspection
We inspect mean absolute error across localities and room sharing types on held-out test data.

In [6]:
print('Mean Absolute Error by Room Type (Held-out Test Data):')
room_err = test_eval_df.groupby('room_type')['abs_error'].agg(['count', 'mean']).round(2)
room_err.columns = ['Sample Count', 'Mean Abs Error (INR)']
print(room_err)

print('\nMean Absolute Error by Locality (Top 5 Test Groups):')
loc_err = test_eval_df.groupby('locality')['abs_error'].agg(['count', 'mean']).round(2)
loc_err.columns = ['Sample Count', 'Mean Abs Error (INR)']
print(loc_err.sort_values(by='Sample Count', ascending=False).head(5))

## 6. Transparent Rule-Based Safety & Warning Engine
We evaluate listings using transparent, explainable checks:
1. **Elevated Deposit Ratio**: Deposit $\ge 4.5\times$ monthly rent.
2. **Advance Payment Prompt**: Demands token or fees prior to physical in-person viewing.
3. **Feature Discrepancy**: Contradictions between textual claims and structured attributes.
4. **Template Descriptions**: Generic bulk boilerplate texts.

In [7]:
sample_listing = {
    'locality': 'Indiranagar',
    'monthly_rent_inr': 18000,
    'deposit_inr': 90000,
    'room_type': 'Single',
    'ac': 'No',
    'pay_before_visit_indicator': 1,
    'listing_description_demo': 'Single PG with AC. Please pay 1000 token before visiting.'
}

safety_report = evaluate_listing_safety(sample_listing, predicted_rent=17500)
print(f"Warning Count: {safety_report['warning_count']} | Status: {safety_report['rule_based_warning_label']}")
print('\nFlagged Verification Indicators:')
for w in safety_report['warnings']:
    print(f" - [{w['severity']}] {w['title']}: {w['message']}")

## 7. Recommendation Engine Demonstration
As outlined in Section 13 of the project plan:
A user selects **BTM Layout**, **double sharing**, **Wi-Fi required**, and a budget of **₹12,000**.

In [8]:
recommender = PGRecommender(cleaned_df)

user_query = {
    'locality': 'BTM Layout',
    'room_type': 'Double Sharing',
    'wifi_required': True,
    'budget': 12000,
    'preferred_amenities': ['Power backup', 'Housekeeping']
}

rec_results = recommender.recommend(user_query, top_k=5)
print(f"Total Matches in Database: {rec_results['total_matches']}\n")

for i, rec in enumerate(rec_results['recommendations'], 1):
    print(f"Option #{i}: {rec['listing_id']} - Rent: INR {rec['monthly_rent_inr']:,} | Deposit: INR {rec['deposit_inr']:,}")
    print(f"  Match Score: {rec['match_score']}/100 | Market Evaluation: {rec['price_tag']}")
    print(f"  Safety Status: {rec['safety']['rule_based_warning_label']}")
    if rec['safety']['warnings']:
        for w in rec['safety']['warnings']:
            print(f"   * Warning: {w['title']} - {w['message']}")
    print()
    
if not rec_results['recommendations']:
    print(rec_results['message'])

## 8. Discussion, Limitations & Ethical Guidelines

### Model & Data Limitations
1. **Demonstration Dataset Scope**: The underlying dataset consists of 400 demonstration records with controlled noise. Performance metrics validate the data science and engineering pipeline, but must not be cited as empirical truths about the Bengaluru real estate market without verified market data.
2. **Post-Outcome Leakage Prevention**: We deliberately excluded deposit from the input feature set for rent estimation. In practical rental journeys, rent is estimated prior to deposit negotiation.
3. **Ethical Limits on Fraud Detection**: A low review rating or high deposit ratio does not prove fraud. Genuine properties may have higher security deposits due to premium electronics. Transparent warnings serve to empower tenants during physical verification.